In [44]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, expr, regexp_replace, split, size, slice,
    lower, trim, when, year, month, dayofweek, hour, length,
    udf, lit, concat_ws, substring_index, max as spark_max
)
from pyspark.sql.types import *

# ----------------------------------------------------------------------------------
# 1. Spark session + logging
# ----------------------------------------------------------------------------------
spark = (
    SparkSession.builder
        .appName("CleanGitHubCommits")
        .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
log = spark._jvm.org.apache.log4j.LogManager.getLogger("CleanGitHubCommits")

In [45]:
# ----------------------------------------------------------------------------------
# 2. Schema – only keep what we actually need
# ----------------------------------------------------------------------------------
file_schema = StructType([
    StructField("filename", StringType(), True),
    StructField("status", StringType(), True),
    StructField("additions", IntegerType(), True),
    StructField("deletions", IntegerType(), True)
])

commit_schema = StructType([
    StructField("sha", StringType(), True),
    StructField("commit", StructType([
        StructField("author", StructType([
            StructField("name", StringType(), True),
            StructField("email", StringType(), True),
            StructField("date", StringType(), True)
        ]), True),
        StructField("committer", StructType([
            StructField("name", StringType(), True),
            StructField("email", StringType(), True),
            StructField("date", StringType(), True)
        ]), True),
        StructField("message", StringType(), True)
    ]), True),
    StructField("html_url", StringType(), True),
    StructField("files", ArrayType(file_schema), True)
])

# ----------------------------------------------------------------------------------
# 3. Load raw Kafka messages
# ----------------------------------------------------------------------------------
df_raw = (
    spark.read
         .format("kafka")
         .option("kafka.bootstrap.servers", "kafka:29092")
         .option("subscribe", "github-commits")
         .option("startingOffsets", "earliest")
         .option("endingOffsets", "latest")
         .load()
)
count = df_raw.count()
print(f"Successfully loaded {count} records from topic 'github-commits'.")



Successfully loaded 242 records from topic 'github-commits'.


In [46]:

# Kafka’s `value` is binary; cast to string and parse JSON
# Step 3: Parse and drop malformed/null-key rows early
df_json = (
    df_raw
      .select(from_json(col("value").cast("string"), commit_schema).alias("c"))
      .select("c.*")
      .filter(
          col("commit").isNotNull() &
          col("commit.message").isNotNull() &
          col("files").isNotNull() &
          (size(col("files")) > 0) &
          col("sha").isNotNull() &
          col("html_url").isNotNull()
      )
)

# ----------------------------------------------------------------------------------
# 4. Helper UDFs / expressions
# ----------------------------------------------------------------------------------


# Simple language inference from filename extension
ext2lang = {
    # Scripting / dynamic
    "py": "python",
    "js": "javascript",
    "ts": "typescript",
    "rb": "ruby",
    "php": "php",
    "pl": "perl",
    "sh": "shell",
    "bat": "batch",
    "ps1": "powershell",

    # Compiled languages
    "java": "java",
    "kt": "kotlin",
    "scala": "scala",
    "c": "c",
    "h": "c",  # C header
    "cpp": "cpp",
    "cc": "cpp",
    "cxx": "cpp",
    "hpp": "cpp",  # C++ header
    "cs": "csharp",
    "go": "go",
    "rs": "rust",
    "swift": "swift",
    "m": "objective-c",
    "mm": "objective-c++",

    # Web / Markup / Style
    "html": "html",
    "htm": "html",
    "css": "css",
    "scss": "scss",
    "less": "less",
    "json": "json",
    "xml": "xml",
    "yaml": "yaml",
    "yml": "yaml",
    "md": "markdown",

    # Data formats / config
    "toml": "toml",
    "ini": "ini",
    "cfg": "config",

    # Others
    "sql": "sql",
    "dart": "dart",
    "lua": "lua",
    "coffee": "coffeescript",
    "vue": "vue",
    "jsx": "jsx",
    "tsx": "tsx",
    "r": "r",
    "jl": "julia",
    "erl": "erlang",
    "ex": "elixir",
    "exs": "elixir",
    "groovy": "groovy",
    "bat": "batch",
    "dockerfile": "dockerfile",  # special filename, might need separate handling
}


# Change‑type heuristic
change_expr = (
    when(lower(col("commit_message_clean")).rlike(r"\bfix(e[ds])?\b|bug"), "fix")
    .when(lower(col("commit_message_clean")).rlike(r"\brefactor|clean"), "refactor")
    .when(lower(col("commit_message_clean")).rlike(r"\b(add|feature|implement)"), "feature")
    .otherwise("other")
)




In [47]:

@udf("string")
def infer_lang(filename):
    if not filename:
        return None
    ext = filename.rsplit('.', 1)[-1].lower() if '.' in filename else ""
    # Basic filename-based heuristic for known extensionless files
    if filename.lower() in {"makefile", "dockerfile", "license"}:
        return filename.lower()
    return ext2lang.get(ext, "Other")

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, BooleanType

def is_bot(name_or_email):
    if name_or_email is None:
        return False
    val = name_or_email.lower()
    keywords = ["noreply", "bot", "ci", "automation", "auto", "deploy"]
    return any(k in val for k in keywords)

is_bot_udf = udf(is_bot, BooleanType())


# ----------------------------------------------------------------------------------
# 5. Flatten to file‑level rows
# ----------------------------------------------------------------------------------

df_exp = (
    df_json
      .withColumn("repo",             substring_index(col("html_url"), "/", -2))
      .withColumn("author_name_raw",  col("commit.author.name"))
      .withColumn("author_email_raw", col("commit.author.email"))
      .withColumn("committer_name",   col("commit.committer.name"))
      .withColumn("committer_email",  col("commit.committer.email"))
      .withColumn("commit_timestamp", col("commit.author.date").cast("timestamp"))
      .withColumn("commit_message_raw", col("commit.message"))
      .withColumn("files",            col("files"))
      .withColumn("author_name",      lower(trim(col("author_name_raw"))))
      .withColumn("author_email",     strip_bot_udf(col("author_email_raw")))
      .drop("author_name_raw", "author_email_raw")
      .withColumn("commit_message_clean",
                  lower(regexp_replace(col("commit_message_raw"), r"http\S+|\p{So}", "")))
      .withColumn("year",        year("commit_timestamp"))
      .withColumn("month",       month("commit_timestamp"))
      .withColumn("day_of_week", dayofweek("commit_timestamp") - 2)
      .withColumn("hour_of_day", hour("commit_timestamp"))
      .withColumn("file",        expr("explode(files)"))
      .selectExpr(
          "repo",
          "sha as commit_sha",
          "author_name",
          "author_email",
          "committer_name",
          "committer_email",
          "file.filename as file_path",
          "file.status as change_status",
          "file.additions as lines_added",
          "file.deletions as lines_deleted",
          "commit_message_clean",
          "commit_timestamp",
          "year","month","day_of_week","hour_of_day"
      )
      .filter(col("file_path").isNotNull())  # extra guard
      .filter(
        (~is_bot_udf(col("author_name"))) &
        (~is_bot_udf(col("author_email"))) &
        (~is_bot_udf(col("committer_name"))) &
        (~is_bot_udf(col("committer_email")))
    )
)
df_exp = df_exp.filter(col("file_path").isNotNull() & (col("file_path") != ""))

# ----------------------------------------------------------------------------------
# 6. File‑level enrichments
# ----------------------------------------------------------------------------------
df_clean = (
    df_exp
.withColumn("file_name", expr("element_at(split(file_path, '/'), -1)"))
      .withColumn("path_depth", size(split(col("file_path"), "/")) - 1)
      .withColumn("module_candidate",
                  concat_ws("/",
                            slice(split(col("file_path"), "/"), 1, 2)))  # first 1‑2 folders
      .withColumn("language", infer_lang(col("file_name")))
      .withColumn("commit_size", col("lines_added") + col("lines_deleted"))
      .withColumn("change_type", change_expr)
      .dropDuplicates(["commit_sha", "file_path"])  # defensive
)

print(df_clean.select("file_name").distinct().show(50, truncate=False))


+----------------------------------------------------------------+
|file_name                                                       |
+----------------------------------------------------------------+
|0098_change_productmedia_verbose_text.py                        |
|PDFDocPrintingWorkpackageProcessor.java                         |
|DocumentInterfaceWrapperHelper.java                             |
|AsyncBatchBL.java                                               |
|checkInvoiceCandStatus.feature                                  |
|DefaultDunningProducer.java                                     |
|compensationGroup.feature                                       |
|FactAcctToTabularStringConverter.java                           |
|C_OLCandToOrderEnqueuer.java                                    |
|djangojs.po                                                     |
|C_Dunning_Candidate_Process_AutomaticallyPDFPrinting.java       |
|test_orders.py                                               

In [48]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, split, slice, concat_ws, unix_timestamp, lit
from pyspark.sql.window import Window
from math import log2



from pyspark.sql.functions import count, row_number
from pyspark.sql.window import Window

# Filter files that are deep enough to belong to a module
df_clean_filtered = df_clean.filter(F.size(F.split(col("file_path"), "/")) > MODULE_DEPTH - 1)

# Extract module_candidate from path
df_clean_filtered = df_clean_filtered.withColumn(
    "module_candidate",
    concat_ws("/", slice(split(col("file_path"), "/"), 1, MODULE_DEPTH))
)

# Filter to folders and re-extract module_candidate if needed
df_lang_per_file = df_clean_filtered.select("module_candidate", "language")

# Count (module, language) occurrences
df_lang_counts = (
    df_lang_per_file
      .groupBy("module_candidate", "language")
      .agg(count("*").alias("lang_count"))
)

# Window to pick most frequent language per module
w_lang = Window.partitionBy("module_candidate").orderBy(col("lang_count").desc())

# Pick dominant language
df_dominant_lang = (
    df_lang_counts
      .withColumn("rank", row_number().over(w_lang))
      .filter(col("rank") == 1)
      .select("module_candidate", col("language").alias("dominant_language"))
)

df_dominant_lang.show(10, truncate=False)


MODULE_DEPTH = 2  # adjust for desired folder granularity

# Build df_module_base with required metadata per module_candidate
df_module_base = (
    df_clean_filtered
      .groupBy("module_candidate")
      .agg(
          F.countDistinct("author_email").alias("num_contributors"),
          F.countDistinct("commit_sha").alias("num_commits"),
          F.sum(F.col("lines_added") + F.col("lines_deleted")).alias("total_churn"),
          F.min("commit_timestamp").alias("first_commit_time"),
          F.max("commit_timestamp").alias("last_commit_time")
      )
)


# Map each (commit, module) pair
df_commit_module = (
    df_clean_filtered
      .select("commit_sha", "module_candidate")
      .distinct()
)

# Count modules per commit and compute entropy contribution
w_commit = Window.partitionBy("commit_sha")

df_entropy_piece = (
    df_commit_module
      .withColumn("modules_in_commit", F.count("*").over(w_commit))
      .withColumn(
          "entropy_piece",
          F.when(F.col("modules_in_commit") == 1, F.lit(0.0))
           .otherwise(
               - (1 / F.col("modules_in_commit")) *
                 F.log2(1 / F.col("modules_in_commit"))
           )
      )
)

# Aggregate per module: avg entropy, commit count, density
w_module = Window.partitionBy("module_candidate")

df_entropy_density = (
    df_entropy_piece
      .withColumn("entropy_sum", F.sum("entropy_piece").over(w_module))
      .withColumn("commit_cnt",  F.count("*").over(w_module))
      .withColumn(
          "density_numer",
          F.sum(F.when(F.col("modules_in_commit") == 1, 1).otherwise(0)).over(w_module)
      )
      .select("module_candidate", "entropy_sum", "commit_cnt", "density_numer")
      .distinct()
      .withColumn("avg_entropy", F.col("entropy_sum") / F.col("commit_cnt"))
      .withColumn("density",     F.col("density_numer") / F.col("commit_cnt"))
      .drop("entropy_sum", "density_numer")
)

# Compute normalization base for entropy (log2 of number of distinct modules)
num_modules = df_commit_module.select("module_candidate").distinct().count()
entropy_norm_base = log2(num_modules) if num_modules > 1 else 1  # avoid div-by-zero

df_entropy_density = df_entropy_density.withColumn(
    "norm_entropy",
    F.when(F.col("avg_entropy") == 0, F.lit(0.0))
     .otherwise(F.col("avg_entropy") / F.lit(entropy_norm_base))
)



df_modules = (
    df_module_base
      .join(df_entropy_density, on="module_candidate", how="left")
      .join(df_dominant_lang, on="module_candidate", how="left")
      .withColumn("age_days", (
          unix_timestamp("last_commit_time") - unix_timestamp("first_commit_time")
      ) / 86400)
      .withColumn("single_author_module", (col("num_contributors") == 1))
      .withColumn("popularity_score", 0.5 * col("num_contributors") + 0.5 * col("num_commits"))
      .withColumn("stability_score", 1 / (1 + col("total_churn")))
      .withColumn("maturity_score", col("age_days") / (col("age_days") + 30))
      .withColumn("cohesion_score",
                  (1 - col("norm_entropy")) * lit(0.7) + col("density") * lit(0.3))
      .drop("avg_entropy", "norm_entropy", "density")
)

# Summary
pd_summary = df_modules.describe().toPandas().set_index("summary").round(2)
from pyspark.sql.functions import col

other_count = df_dominant_lang.filter(col("dominant_language") == "Other").count()

print(f"Number of modules with 'Other' dominant language: {other_count/df_dominant_lang.count()*100}")


+------------------------------------+-----------------+
|module_candidate                    |dominant_language|
+------------------------------------+-----------------+
|app/controllers                     |ruby             |
|app/helpers                         |ruby             |
|app/javascript                      |javascript       |
|app/models                          |ruby             |
|app/views                           |Other            |
|backend/de-metas-salesorder         |java             |
|backend/de.metas.adempiere.adempiere|java             |
|backend/de.metas.async              |java             |
|backend/de.metas.business           |java             |
|backend/de.metas.cucumber           |java             |
+------------------------------------+-----------------+
only showing top 10 rows



Number of modules with 'Other' dominant language: 41.07142857142857


In [29]:
print(pd_summary)

        module_candidate    num_contributors         num_commits  \
summary                                                            
count                 66                  66                  66   
mean                None  1.2727272727272727  3.5757575757575757   
stddev              None  0.7135060680126758    3.88314149610697   
min        .cursor/rules                   0                   1   
max          test/system                   4                  25   

               total_churn          commit_cnt dominant_language  \
summary                                                            
count                   66                  66                 0   
mean     440.1060606060606  3.5757575757575757              None   
stddev    826.183498499863    3.88314149610697              None   
min                      4                   1              None   
max                   4412                  25              None   

                   age_days    popularity_scor

In [49]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

N_DAYS = 90  # window for recent contributor flag

# 1. Aggregate contributions per author per module
df_contrib = (
    df_clean_filtered
      .groupBy("module_candidate", "author_name")
      .agg(
          F.count("commit_sha").alias("contributions"),
          F.max("commit_timestamp").alias("last_commit")
      )
      .withColumnRenamed("module_candidate", "module_path")
)

# 2. Join dominant language info to each module_path
df_contrib_lang = df_contrib.join(
    df_dominant_lang,
    df_contrib.module_path == df_dominant_lang.module_candidate,
    how="left"
).drop("module_candidate")

# 3. Calculate contributor ranking within each module based on contributions
w_module = Window.partitionBy("module_path").orderBy(F.desc("contributions"))

df_contrib_ranked = (
    df_contrib_lang
      .withColumn("rank", F.row_number().over(w_module))
      .withColumn("primary_author", (F.col("rank") == 1))
      .withColumn(
          "recent_contributor",
          F.datediff(F.current_timestamp(), F.col("last_commit")) <= N_DAYS
      )
)

# 4. Aggregate per author and language for skill summary across all modules
df_author_lang_skill = (
    df_contrib_ranked
      .groupBy("author_name", "dominant_language")
      .agg(
          F.sum("contributions").alias("total_contributions"),
          F.max("last_commit").alias("last_commit")
      )
      .withColumn(
          "recent_contributor",
          F.datediff(F.current_timestamp(), F.col("last_commit")) <= N_DAYS
      )
)

# 5. Outputs:

# Show contributors per module with their rank and flags
print("=== Contributors per Module ===")
df_contrib_ranked.select(
    "module_path",
    "author_name",
    "dominant_language",
    "contributions",
    "last_commit",
    "primary_author",
    "recent_contributor",
    "rank"
).orderBy("module_path", "rank").show(truncate=False, n=50)

# Show skill summary per author and language
print("=== Skill Summary per Author-Language ===")
df_author_lang_skill.select(
    "author_name",
    "dominant_language",
    "total_contributions",
    "last_commit",
    "recent_contributor"
).orderBy("author_name", "dominant_language").show(truncate=False, n=50)



=== Contributors per Module ===


+------------------------------------+---------------+-----------------+-------------+-------------------+--------------+------------------+----+
|module_path                         |author_name    |dominant_language|contributions|last_commit        |primary_author|recent_contributor|rank|
+------------------------------------+---------------+-----------------+-------------+-------------------+--------------+------------------+----+
|app/controllers                     |josh pigford   |ruby             |3            |2025-06-18 13:28:32|true          |true              |1   |
|app/helpers                         |zach gollwitzer|ruby             |1            |2025-06-30 14:22:37|true          |true              |1   |
|app/javascript                      |zach gollwitzer|javascript       |1            |2025-06-27 16:57:23|true          |true              |1   |
|app/models                          |josh pigford   |ruby             |4            |2025-06-18 13:28:32|true          |tru

[Stage 819:>                (0 + 1) / 1][Stage 820:>                (0 + 1) / 1]

+---------------+-----------------+-------------------+-------------------+------------------+
|author_name    |dominant_language|total_contributions|last_commit        |recent_contributor|
+---------------+-----------------+-------------------+-------------------+------------------+
|adrian         |java             |117                |2025-03-27 12:53:58|false             |
|adrian         |python           |1                  |2025-03-27 11:20:13|false             |
|bill kronholm  |python           |4                  |2021-08-05 22:31:10|false             |
|bill-shuup     |python           |3                  |2021-08-16 17:32:34|false             |
|christian hess |Other            |56                 |2021-08-17 18:28:47|false             |
|christian hess |python           |25                 |2021-08-17 13:24:56|false             |
|josh pigford   |Other            |14                 |2025-06-18 10:38:23|true              |
|josh pigford   |ruby             |31             